# Agent Platform pipelines continuous training

### Load configuration settings from the setup notebook

Set the constants used in this notebook and load the config settings from the `00_environment_setup.ipynb` notebook.

In [ ]:
GCP_PROJECTS = !gcloud config get-value project
PROJECT_ID = GCP_PROJECTS[0]
BUCKET_NAME = f"{PROJECT_ID}-fraudfinder"
config = !gsutil cat gs://{BUCKET_NAME}/config/notebook_env.py
print(config.n)
exec(config.n)

### Get Default Service account

In [ ]:
GCP_SRV_ACC = !gcloud iam service-accounts list \
    --filter="email ~ -compute@developer.gserviceaccount.com" \
    --format="value(email)"
GCP_SERV_ACCOUNT=GCP_SRV_ACC[0]
print(GCP_SERV_ACCOUNT)

### Import libraries and define constants

#### Libraries
Here, we'll import the necessary libraries for this notebook.

In [ ]:
# General
import datetime as dt
import json
import os
import random
import sys
import time
from datetime import datetime, timedelta
from typing import List, Union

# Data Engineering
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 500)

# Vertex AI and Vertex AI Feature Store
from google.cloud import aiplatform
from google.cloud import bigquery

### Get Endpoint ID and current deployed model ID

In [ ]:
endpoints = aiplatform.Endpoint.list(
    filter=f"display_name={ENDPOINT_NAME}",  # optional: filter by specific endpoint name
    order_by="update_time",
)

target_endpoint = endpoints[-1]
ENDPOINT_ID = target_endpoint.name
print(f"ENDPOINT_ID: {ENDPOINT_ID}")

# Access the underlying GAPIC resource to check for deployed models
if target_endpoint.gca_resource.deployed_models:
    # Retrieve the ID of the first deployed model
    DEPLOYED_MODEL_ID = target_endpoint.gca_resource.deployed_models[0].id
    print(f"DEPLOYED_MODEL_ID: {DEPLOYED_MODEL_ID}")
else:
    print("No models are currently deployed to this endpoint.")

### Create TensorBoard instance

In [ ]:
from google.cloud import aiplatform

TB_DISPLAY_NAME = "fraud-model-tensorboard"

# 1. Search for existing TensorBoard instances with this display name
existing_tensorboards = aiplatform.Tensorboard.list(
    filter=f"display_name={TB_DISPLAY_NAME}",
    project=PROJECT_ID,
    location=REGION,
)

# 2. Check if the instance already exists
if existing_tensorboards:
    print(f"Found existing TensorBoard instance: {TB_DISPLAY_NAME}")
    # Grab the first matched instance from the list
    tensorboard = existing_tensorboards[0]
else:
    print(f"TensorBoard instance '{TB_DISPLAY_NAME}' not found. Creating a new one...")
    # Create the TensorBoard instance
    tensorboard = aiplatform.Tensorboard.create(
        display_name=TB_DISPLAY_NAME,
        project=PROJECT_ID,
        location=REGION,
    )

# 3. Grab the resource name string to pass to your KFP job
TB_RESOURCE_NAME = tensorboard.resource_name
print(f"TB_RESOURCE_NAME: {TB_RESOURCE_NAME}")

## Orchestrating a training pipeline with Vertex Pipelines

In [ ]:
import os
from kfp import dsl
from kfp.dsl import (
    component,
    Input,
    Output,
    Artifact,
    Model,
    Metrics,
    ClassificationMetrics,
    HTML,
)
from typing import NamedTuple
# Vertex AI Components
from google_cloud_pipeline_components.types import artifact_types
from google_cloud_pipeline_components.v1.model import ModelGetOp, ModelUploadOp
from google_cloud_pipeline_components.v1.custom_job import create_custom_training_job_op_from_component
from google_cloud_pipeline_components.v1.batch_predict_job import ModelBatchPredictOp
from google_cloud_pipeline_components.v1.dataset import TabularDatasetCreateOp
from google_cloud_pipeline_components.v1.bigquery import BigqueryQueryJobOp
from google_cloud_pipeline_components.v1.endpoint import ModelDeployOp
from google_cloud_pipeline_components.v1.endpoint import ModelUndeployOp

# =====================================================================
# 1. Custom Components
# =====================================================================

@component(base_image="python:3.11-slim")
def prep_dataset_statement_op(
    bq_source_table: str,
    train_partition: str,
    export_format: str,
    downsample_majority_class: bool,
    gcs_data_export_dir: str,
    run_id: str,
) -> NamedTuple("Outputs", [
        ("dataset_export_path", str),
        ("dataset_export_sql", str),
    ]):

    # Formulate the dynamic target path for the BigQuery export
    dataset_export_path = f"{gcs_data_export_dir}/{run_id}/{train_partition}/*.{export_format}"

    # Downsample to 5% for majority class 
    downsample_majority_class_sql = f"""
    AND tx_fraud = 1
    OR (tx_fraud = 0 AND MOD(ABS(FARM_FINGERPRINT(CAST(transaction_id AS STRING))), 100) < 5)
    """ if downsample_majority_class else ""

    header_option = ", header=true" if export_format=='csv' else ""

    dataset_export_sql = f"""
    EXPORT DATA OPTIONS(
      uri='{dataset_export_path}',
      format='{export_format}'{header_option}
    ) AS
    SELECT 
        transaction_id,
        tx_fraud,
        tx_amount,
        customer_id_nb_tx_14day_window, customer_id_nb_tx_7day_window, customer_id_nb_tx_1day_window,
        customer_id_avg_amount_14day_window, customer_id_avg_amount_7day_window, customer_id_avg_amount_1day_window,
        terminal_id_nb_tx_14day_window, terminal_id_nb_tx_7day_window, terminal_id_nb_tx_1day_window,
        terminal_id_risk_14day_window, terminal_id_risk_7day_window, terminal_id_risk_1day_window,
        customer_id_nb_tx_60min_window, customer_id_nb_tx_30min_window, customer_id_nb_tx_15min_window,
        customer_id_avg_amount_60min_window, customer_id_avg_amount_30min_window, customer_id_avg_amount_15min_window,
        terminal_id_nb_tx_60min_window, terminal_id_nb_tx_30min_window, terminal_id_nb_tx_15min_window,
        terminal_id_avg_amount_60min_window, terminal_id_avg_amount_30min_window, terminal_id_avg_amount_15min_window
    FROM `{bq_source_table}`
    WHERE 
        train_split='{train_partition}' {downsample_majority_class_sql}
    """

    return dataset_export_path, dataset_export_sql


@component(base_image="python:3.11-slim")
def compare_models_op(
    base_model_metrics: Input[Metrics],
    new_model_metrics: Input[Metrics],
    metric_name: str,
) -> bool:
    val_base = base_model_metrics.metadata.get(metric_name)
    val_new = new_model_metrics.metadata.get(metric_name)

    if val_base is None or val_new is None:
        raise ValueError(f"Metric '{metric_name}' missing from evaluation logs.")

    val_base = float(val_base)
    val_new = float(val_new)

    print(f"Base Model '{metric_name}': {val_base}")
    print(f"New Model '{metric_name}': {val_new}")

    #return val_new > val_base #TODO: uncomment this
    return True

@component(
    base_image="python:3.11-slim",
    packages_to_install=["pandas", "gcsfs", "scikit-learn", "numpy"]
)
def validate_model_predictions_op(
    truth_gcs_pattern: str,
    preds_gcs_artifact: Input[Artifact],
    evaluation_metrics: Output[ClassificationMetrics],
    scalar_metrics: Output[Metrics]
):
    import pandas as pd
    import numpy as np
    import gcsfs
    import json
    from sklearn.metrics import (
        roc_auc_score, average_precision_score, precision_score, 
        recall_score, f1_score, confusion_matrix, roc_curve
    )

    preds_gcs_pattern = f"{preds_gcs_artifact.uri}/prediction.results*"

    def load_jsonl_from_gcs(uri_pattern):
        fs = gcsfs.GCSFileSystem()
        matched_files = fs.glob(uri_pattern)
        if not matched_files:
            raise FileNotFoundError(f"No files matched: {uri_pattern}")
        return pd.concat([pd.read_json(f"gs://{fp}", lines=True) for fp in matched_files], ignore_index=True)

    df_truth = load_jsonl_from_gcs(truth_gcs_pattern)
    df_preds = load_jsonl_from_gcs(preds_gcs_pattern)

    df_truth = df_truth[['transaction_id', 'tx_fraud']].copy()
    df_truth['tx_fraud'] = df_truth['tx_fraud'].astype(int)

    def parse_predictions(prediction_dict):
        classes = prediction_dict.get('classes', [])
        scores = prediction_dict.get('scores', [])
        try:
            fraud_index = classes.index("1")
            max_score_index = scores.index(max(scores))
            return scores[fraud_index], int(classes[max_score_index])
        except ValueError:
            return 0.0, 0

    parsed_results = df_preds['prediction'].apply(parse_predictions)
    df_preds['fraud_prob'] = parsed_results.apply(lambda x: x[0])
    df_preds['pred_class'] = parsed_results.apply(lambda x: x[1])

    df_merged = pd.merge(df_truth, df_preds, on='transaction_id', how='inner')
    y_true = df_merged['tx_fraud'].tolist()
    y_prob = df_merged['fraud_prob'].tolist()
    y_pred = df_merged['pred_class'].tolist()
    
    cm = confusion_matrix(y_true, y_pred)
    evaluation_metrics.log_confusion_matrix(categories=["Legitimate", "Fraud"], matrix=cm.tolist())

    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    thresholds = np.where(np.isinf(thresholds), 1.0, thresholds)
    
    # Downsample the ROC curve points to avoid exceeding the 128 KB executor_output.json limit
    step = max(1, len(fpr) // 100) 
    evaluation_metrics.log_roc_curve(
        fpr=fpr[::step].tolist(), 
        tpr=tpr[::step].tolist(), 
        threshold=thresholds[::step].tolist()
    )

    metrics_dict = {
        "auRoc": float(roc_auc_score(y_true, y_prob)),
        "auPrc": float(average_precision_score(y_true, y_prob)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1Score": float(f1_score(y_true, y_pred, zero_division=0))
    }
    
    for key, value in metrics_dict.items():
        scalar_metrics.log_metric(key, value)
        
    with open(scalar_metrics.path, 'w') as f:
        json.dump(metrics_dict, f)

@component( 
    base_image="us-docker.pkg.dev/vertex-ai/training/tf-cpu.2-17.py310:latest",
    packages_to_install=["scikit-learn", "pandas", "pyarrow", "db-dtypes"]
)
def train_model(
    project_id: str,          
    model_output_uri: str,
    train_files_pattern: str,
    validation_files_pattern: str,
    metrics: Output[Metrics],
    classification_metrics: Output[ClassificationMetrics]
):
    import os
    os.environ["KERAS_BACKEND"] = "tensorflow"
    import keras
    import numpy as np
    import tensorflow as tf
    from sklearn.metrics import roc_curve, precision_recall_curve, auc, confusion_matrix, matthews_corrcoef, f1_score

    CSV_COLUMNS = [
        'transaction_id', 'tx_fraud', 
        'tx_amount',
        'customer_id_nb_tx_14day_window', 'customer_id_nb_tx_7day_window', 'customer_id_nb_tx_1day_window',
        'customer_id_avg_amount_14day_window', 'customer_id_avg_amount_7day_window', 'customer_id_avg_amount_1day_window',
        'terminal_id_nb_tx_14day_window', 'terminal_id_nb_tx_7day_window', 'terminal_id_nb_tx_1day_window',
        'terminal_id_risk_14day_window', 'terminal_id_risk_7day_window', 'terminal_id_risk_1day_window',
        'customer_id_nb_tx_60min_window', 'customer_id_nb_tx_30min_window', 'customer_id_nb_tx_15min_window',
        'customer_id_avg_amount_60min_window', 'customer_id_avg_amount_30min_window', 'customer_id_avg_amount_15min_window',
        'terminal_id_nb_tx_60min_window', 'terminal_id_nb_tx_30min_window', 'terminal_id_nb_tx_15min_window',
        'terminal_id_avg_amount_60min_window', 'terminal_id_avg_amount_30min_window', 'terminal_id_avg_amount_15min_window'
    ]

    target_column_name='tx_fraud'
    features_columns_offest = 2
    input_feature_names = CSV_COLUMNS[features_columns_offest:] 
    RECORD_DEFAULTS = [[''], [0]] + [[0.0]] * (len(CSV_COLUMNS) - features_columns_offest)
    
    def parse_csv_row(csv_row):
        columns = tf.io.decode_csv(csv_row, record_defaults=RECORD_DEFAULTS)
        features = dict(zip(CSV_COLUMNS, columns))

        #remove transaction_id from features
        features.pop('transaction_id')

        #remove target from features
        label = tf.expand_dims(features.pop(target_column_name), axis=-1)
        
        for key in features.keys():
            features[key] = tf.expand_dims(features[key], axis=-1)
        return features, label

    batch_size = 256
    epochs = 10
    steps_per_epoch=100
    learning_rate=0.001
    
    train_files = tf.data.Dataset.list_files(train_files_pattern)
    train_dataset = train_files.interleave(
        lambda filename: tf.data.TextLineDataset(filename).skip(1),
        cycle_length=tf.data.AUTOTUNE,
        num_parallel_calls=tf.data.AUTOTUNE
    ).shuffle(buffer_size=1000).batch(batch_size).map(parse_csv_row, num_parallel_calls=tf.data.AUTOTUNE).repeat().prefetch(tf.data.AUTOTUNE)

    validation_files = tf.data.Dataset.list_files(validation_files_pattern)
    validation_dataset = validation_files.interleave(
        lambda filename: tf.data.TextLineDataset(filename).skip(1),
        cycle_length=tf.data.AUTOTUNE,
        num_parallel_calls=tf.data.AUTOTUNE
    ).batch(batch_size).map(parse_csv_row, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)

    inputs = [keras.Input(shape=(1,), name=name) for name in input_feature_names]
    x = keras.layers.concatenate(inputs)
    x = keras.layers.Dense(32, activation="relu")(x)
    x = keras.layers.Dense(16, activation="relu")(x)
    x = keras.layers.Dense(1, activation="sigmoid")(x)
    model = keras.Model(inputs=inputs, outputs=x)

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss=keras.losses.BinaryCrossentropy(),
        metrics=[keras.metrics.BinaryAccuracy(), 
                 keras.metrics.AUC(curve="ROC", name="auc"), 
                 keras.metrics.AUC(curve="PR", name="pr")]
    )

    tensorboard_log_dir = os.environ.get("AIP_TENSORBOARD_LOG_DIR", "/tmp/tensorboard")
    
    callbacks = [
        keras.callbacks.TensorBoard(log_dir=tensorboard_log_dir, histogram_freq=1, write_graph=True),
        keras.callbacks.EarlyStopping(monitor='val_pr', mode='max', patience=5, restore_best_weights=True)
    ]

    model.fit(
        train_dataset, 
        validation_data=validation_dataset, 
        epochs=epochs, 
        steps_per_epoch=steps_per_epoch, 
        callbacks=callbacks, 
        verbose=1
    )

    y_true, y_pred_probs = [], []
    for features, label in validation_dataset:
        preds = model(features, training=False)
        y_true.extend(label.numpy().flatten())
        y_pred_probs.extend(preds.numpy().flatten())

    y_true = np.array(y_true)
    y_pred_probs = np.array(y_pred_probs)
    y_pred_classes = (y_pred_probs >= 0.5).astype(int)

    fpr, tpr, roc_thresholds = roc_curve(y_true, y_pred_probs)
    precision, recall, pr_thresholds = precision_recall_curve(y_true, y_pred_probs)
    cm = confusion_matrix(y_true, y_pred_classes)

    metrics.log_metric("roc_auc", float(auc(fpr, tpr)))
    metrics.log_metric("pr_auc", float(auc(recall, precision)))
    metrics.log_metric("f1_score", float(f1_score(y_true, y_pred_classes)))
    metrics.log_metric("mcc", float(matthews_corrcoef(y_true, y_pred_classes)))

    step = max(1, len(fpr) // 100) 
    classification_metrics.log_roc_curve(fpr[::step].tolist(), tpr[::step].tolist(), [1.0 if np.isinf(x) else x for x in roc_thresholds[::step].tolist()])
    classification_metrics.log_confusion_matrix(["Legitimate", "Fraudulent"], cm.tolist())

    # Model serving input-ouput wrapper
    signature_dict = {col: tf.TensorSpec(shape=(None,), dtype=tf.float32, name=col) for col in input_feature_names}
    @tf.function(input_signature=[signature_dict])
    def serving_fn(json_inputs):
        prob_1 = model({col: tf.expand_dims(json_inputs[col], axis=1) for col in input_feature_names})
        prob_0 = 1.0 - prob_1
        return {"scores": tf.concat([prob_0, prob_1], axis=1), "classes": tf.tile(tf.constant([['0', '1']]), [tf.shape(prob_1)[0], 1])}
    
    export_archive = keras.export.ExportArchive()
    export_archive.track(model)
    export_archive.add_endpoint(name="serving_default", fn=serving_fn)
    export_archive.write_out(model_output_uri)

# =====================================================================
# 2. Custom Job Op Creation
# =====================================================================
train_model_custom_job = create_custom_training_job_op_from_component(
    component_spec=train_model,
    display_name="fraud-custom-model-training-job",
    machine_type="e2-standard-8", 
)

# =====================================================================
# 3. Pipeline Graph
# =====================================================================
@dsl.pipeline(
    name="tx-fraud-retrain-pipeline",
    description="Continuous training with Keras fraud model"
)
def fraud_pipeline(
    project_id: str,
    location: str,
    service_account: str,
    
    # BigQuery Inputs
    bq_source_table: str,
    
    # GCS Data Paths
    gcs_data_export_dir: str,
    # gcs_test_csv_pattern: str,
    # gcs_eval_jsonl_pattern: str,
    
    # GCS Output Paths
    gcs_model_staging_dir: str,
    gcs_predict_output_dir: str,
    
    # Vertex AI Resources
    endpoint_id: str,
    deployed_model_id: str,
    tensorboard_resource_name: str,
    serving_container_image_uri: str = "us-docker.pkg.dev/vertex-ai/prediction/tf2-cpu.2-15:latest",
    
    enable_caching: bool = False,
):
    # This automatically becomes a unique ID on EVERY scheduled run
    run_id = dsl.PIPELINE_JOB_ID_PLACEHOLDER
    
    train_dataset_export_statements = prep_dataset_statement_op(
        bq_source_table=bq_source_table,
        train_partition='train',
        export_format='csv',
        downsample_majority_class=True,
        run_id=run_id,
        gcs_data_export_dir=gcs_data_export_dir,
    )

    train_dataset_export = BigqueryQueryJobOp(
        project=project_id,
        location=location,
        query=train_dataset_export_statements.outputs["dataset_export_sql"]
    )
    train_dataset_export.after(train_dataset_export_statements)

    train_dataset_create_task = TabularDatasetCreateOp(
        project=project_id,
        location=location,
        display_name="register_train_dataset",
        gcs_source=train_dataset_export_statements.outputs["dataset_export_path"],
    )
    train_dataset_create_task.after(train_dataset_export)

    validation_dataset_export_statements = prep_dataset_statement_op(
        bq_source_table=bq_source_table,
        train_partition='validation',
        export_format='csv',
        downsample_majority_class=False,
        run_id=run_id,
        gcs_data_export_dir=gcs_data_export_dir,
    )

    validation_dataset_export = BigqueryQueryJobOp(
        project=project_id,
        location=location,
        query=validation_dataset_export_statements.outputs["dataset_export_sql"]
    )
    validation_dataset_export.after(validation_dataset_export_statements)

    validation_dataset_create_task = TabularDatasetCreateOp(
        project=project_id,
        location=location,
        display_name="register_validation_dataset",
        gcs_source=validation_dataset_export_statements.outputs["dataset_export_path"]
    )
    validation_dataset_create_task.after(validation_dataset_export)

    training_task = train_model_custom_job(
        project=project_id, 
        location=location,
        project_id=project_id, 
        model_output_uri=gcs_model_staging_dir,
        train_files_pattern=train_dataset_export_statements.outputs["dataset_export_path"], 
        validation_files_pattern=validation_dataset_export_statements.outputs["dataset_export_path"],
        tensorboard=tensorboard_resource_name,
        service_account=service_account,
        base_output_directory=gcs_model_staging_dir
    )
    training_task.after(train_dataset_create_task, validation_dataset_create_task)

    model_importer = dsl.importer(
        artifact_uri=gcs_model_staging_dir,
        artifact_class=artifact_types.UnmanagedContainerModel,
        metadata={"containerSpec": {"imageUri": serving_container_image_uri}},
    )
    model_importer.after(training_task)

    endpoint_resource_name = f"projects/{project_id}/locations/{location}/endpoints/{endpoint_id}"
    endpoint_uri = f"https://{location}-aiplatform.googleapis.com/v1/{endpoint_resource_name}"

    endpoint_importer = dsl.importer(
        artifact_uri=endpoint_uri,
        artifact_class=artifact_types.VertexEndpoint,
        metadata={"resourceName": endpoint_resource_name}
    )

    get_model_task = ModelGetOp(
        project=project_id,
        location=location,
        model_name=deployed_model_id
    )
    get_model_task.after(model_importer)

    upload_model_task = ModelUploadOp(
        location=location,
        display_name="create_model_new_version",
        parent_model=get_model_task.outputs["model"], 
        unmanaged_container_model=model_importer.outputs["artifact"],
        version_aliases=['challenger']
    )

    test_dataset_export_statements = prep_dataset_statement_op(
        bq_source_table=bq_source_table,
        train_partition='test',
        export_format='json',
        downsample_majority_class=False,
        run_id=run_id,
        gcs_data_export_dir=gcs_data_export_dir,
    )

    test_dataset_export = BigqueryQueryJobOp(
        project=project_id,
        location=location,
        query=test_dataset_export_statements.outputs["dataset_export_sql"]
    )

    batch_base_predict_task = ModelBatchPredictOp(
        project=project_id,
        location=location,
        key_field="transaction_id",
        model=get_model_task.outputs["model"],
        job_display_name="tx-fraud-base-batch-predict",
        instances_format="jsonl",
        gcs_source_uris=[test_dataset_export_statements.outputs["dataset_export_path"]], 
        predictions_format="jsonl",
        gcs_destination_output_uri_prefix=gcs_predict_output_dir,
        machine_type="e2-standard-4",
        excluded_fields=['tx_fraud','transaction_id'],
        instance_type="object",
    )
    batch_base_predict_task.after(test_dataset_export)

    batch_new_predict_task = ModelBatchPredictOp(
        project=project_id,
        location=location,
        key_field="transaction_id",
        model=upload_model_task.outputs["model"],
        job_display_name="tx-fraud-new-batch-predict",
        instances_format="jsonl",
        gcs_source_uris=[test_dataset_export_statements.outputs["dataset_export_path"]], 
        predictions_format="jsonl",
        gcs_destination_output_uri_prefix=gcs_predict_output_dir,
        machine_type="e2-standard-4",
        excluded_fields=['tx_fraud','transaction_id'],
        instance_type="object",
    )
    batch_new_predict_task.after(test_dataset_export)
    
    eval_model_base = validate_model_predictions_op(
        truth_gcs_pattern=test_dataset_export_statements.outputs["dataset_export_path"],
        preds_gcs_artifact=batch_base_predict_task.outputs["gcs_output_directory"]
    )

    eval_model_new = validate_model_predictions_op(
        truth_gcs_pattern=test_dataset_export_statements.outputs["dataset_export_path"],
        preds_gcs_artifact=batch_new_predict_task.outputs["gcs_output_directory"]
    )

    compare_models_task = compare_models_op(
        base_model_metrics=eval_model_base.outputs['scalar_metrics'],
        new_model_metrics=eval_model_new.outputs['scalar_metrics'],
        metric_name="auPrc"
    )

    with dsl.If(compare_models_task.output == True, name="deploy_challanger_model"):
        
        deploy_task = ModelDeployOp(
            model=upload_model_task.outputs["model"], 
            endpoint=endpoint_importer.outputs["artifact"],
            dedicated_resources_machine_type="n1-standard-4",
            dedicated_resources_min_replica_count=1,
            dedicated_resources_max_replica_count=1,
            traffic_split={"0": 100},  
        )

        undeploy_task = ModelUndeployOp(
            model=get_model_task.outputs["model"], 
            endpoint=endpoint_importer.outputs["artifact"], 
        )
        undeploy_task.after(deploy_task)
        

In [ ]:
from google.cloud import aiplatform
from kfp import compiler
from datetime import datetime

# 1. Generate the dynamic timestamp
current_timestamp = datetime.now().strftime("%Y%m%d-%H%M")
print(f"Submitting pipeline for runtime: {current_timestamp}")

# 2. Compile the generic logic once
compiler.Compiler().compile(
    pipeline_func=fraud_pipeline,
    package_path="tx_fraud_pipeline.yaml"
)

# 3. Define the parameters using your original hardcoded values
DEV_PARAMETERS = {    
    # GCP Infrastructure
    "project_id": PROJECT_ID,
    "location": REGION,
    "service_account": GCP_SERV_ACCOUNT,
    
    # BigQuery
    "bq_source_table": "tx.ff_join_tx_features",
    
    # GCS Data Paths
    "gcs_data_export_dir": f"gs://{BUCKET_NAME}/data_export",
    
    # GCS Output Paths
    "gcs_model_staging_dir": f"gs://{BUCKET_NAME}/model/dnn_keras",
    "gcs_predict_output_dir": f"gs://{BUCKET_NAME}/prediction_outputs",
    
    # Vertex AI Resources
    "endpoint_id": ENDPOINT_ID,
    "deployed_model_id": DEPLOYED_MODEL_ID,
    "tensorboard_resource_name": TB_RESOURCE_NAME,
    "serving_container_image_uri": "us-docker.pkg.dev/vertex-ai/prediction/tf2-cpu.2-15:latest",
    
    # Execution options
    "enable_caching": False,
}

# 4. Initialize and submit the Vertex AI Pipeline Job
pipeline_job = aiplatform.PipelineJob(
    display_name=f"tx-fraud-retrain-pipeline",
    template_path="tx_fraud_pipeline.yaml",
    # Set the pipeline root to your bucket
    pipeline_root=f"gs://{BUCKET_NAME}/pipeline_root", 
    parameter_values=DEV_PARAMETERS,
    enable_caching=False 
)

pipeline_job.submit()

### Schedule retraining pipeline

In [ ]:
from google.cloud import aiplatform

# 1. Define the pipeline job exactly as you normally would
# pipeline_job = aiplatform.PipelineJob(
#     display_name=f"tx-fraud-retrain-pipeline",
#     template_path="tx_fraud_pipeline.yaml",
#     # Set the pipeline root to your bucket
#     pipeline_root=f"gs://{BUCKET_NAME}/pipeline_root", 
#     parameter_values=DEV_PARAMETERS,
#     enable_caching=False 
# )

# 2. Wrap the job in a Schedule object
pipeline_schedule = aiplatform.PipelineJobSchedule(
    pipeline_job=pipeline_job,
    display_name="hourly-fraud-retrain-schedule"
)

# 3. Start the schedule using a Cron string
pipeline_schedule.create(
    cron="0 * * * *",              # Runs exactly at the top of every hour
    max_concurrent_run_count=1     # Prevents 2 pipelines from running at once if one takes > 60 mins
)

print(f"Schedule created! View it at: {pipeline_schedule.resource_name}")

Copyright 2026 Google LLC

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    https://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.